# Health Horizon: Chest X-ray Classification Pipeline

> **CRITICAL MEDICAL DISCLAIMER:**
> Health Horizon is strictly an educational and research prototype developed to study computer vision and transfer learning applications on thoracic radiographs.
> **It is NOT a certified medical device, diagnostic tool, or clinical aid, and MUST NOT be used for clinical decision-making or patient diagnosis.**

This notebook demonstrates the end-to-end machine learning pipeline for **Pneumonia vs. No Pneumonia** binary classification using the **NIH ChestX-ray14 dataset** schema with **MobileNetV2** transfer learning.

In [ ]:
import os
import sys
# Prevent potential tensorflow-metal plugin collisions on macOS
_orig_exists = os.path.exists
os.path.exists = lambda p: False if 'tensorflow-plugins' in str(p) else _orig_exists(p)
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

print('TensorFlow Version:', tf.__version__)
print('NumPy Version:     ', np.__version__)
print('Pandas Version:    ', pd.__version__)

## 1. Dataset Preparation & Patient-Level Splitting

In the NIH ChestX-ray14 dataset, patients often have multiple scans taken over time. Splitting randomly at the image level causes severe data leakage because the network memorizes specific patient anatomy rather than pathological patterns.

Our pipeline enforces **patient-level splitting**, guaranteeing that every patient ID appears in only one partition (Train, Validation, or Test).

In [ ]:
from data_loader import create_sample_dataset, load_and_preprocess_metadata, split_data_by_patient

# Generate or verify sample dataset
csv_path, img_dir = create_sample_dataset(output_dir='data/sample', num_patients=16, scans_per_patient=2)
df = load_and_preprocess_metadata(csv_path, target_disease='Pneumonia', image_dir=img_dir)
print(f'Total records loaded: {len(df)}')
print(f'Total unique patients: {df["patient_id"].nunique()}')
print(f'Pneumonia positive cases: {df["label"].sum()} ({df["label"].mean()*100:.1f}%)')

# Split at patient level (70% train, 15% val, 15% test)
train_df, val_df, test_df = split_data_by_patient(df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)

# Verify zero patient overlap
train_patients = set(train_df['patient_id'])
val_patients = set(val_df['patient_id'])
test_patients = set(test_df['patient_id'])

assert len(train_patients.intersection(val_patients)) == 0
assert len(train_patients.intersection(test_patients)) == 0
assert len(val_patients.intersection(test_patients)) == 0
print('Verification Passed: ZERO patient overlap between train, val, and test splits!')

## 2. Visualizing Chest Radiographs

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i in range(4):
    row = df.iloc[i]
    img = Image.open(row['image_path'])
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Patient: {row['patient_id']}\nFinding: {row['finding_labels']}", fontsize=10)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 3. CNN Architecture with Transfer Learning

We use MobileNetV2 (or ResNet50) as the feature extractor, coupled with a custom classification head:
1. `GlobalAveragePooling2D`
2. `Dense(128, activation='relu')`
3. `Dropout(0.4)`
4. `Dense(1, activation='sigmoid')`

In [ ]:
from model import build_chest_xray_model

model = build_chest_xray_model(
    base_model_name='MobileNetV2',
    input_shape=(224, 224, 3),
    num_classes=1,
    dense_units=128,
    dropout_rate=0.4,
    learning_rate=1e-4,
    freeze_base=True,
    weights=None
)
model.summary()

## 4. Model Training & Validation

In [ ]:
from train import train_pipeline

history = train_pipeline(
    metadata_path='data/sample/sample_metadata.csv',
    image_dir='data/sample/images',
    backbone='MobileNetV2',
    epochs=3,
    batch_size=8,
    learning_rate=1e-4,
    output_model_path='checkpoints/best_model.h5',
    history_path='outputs/training_history.json',
    weights=None
)

## 5. Quantitative Evaluation & Metrics

In [ ]:
from evaluate import evaluate_model

metrics = evaluate_model(
    model_path='checkpoints/best_model.h5',
    test_csv='outputs/splits/test.csv',
    history_path='outputs/training_history.json',
    output_dir='outputs',
    threshold=0.5
)

## 6. Diagnostic Plots: Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

if os.path.exists('outputs/confusion_matrix.png'):
    ax1.imshow(Image.open('outputs/confusion_matrix.png'))
    ax1.set_title('Confusion Matrix')
    ax1.axis('off')

if os.path.exists('outputs/roc_curve.png'):
    ax2.imshow(Image.open('outputs/roc_curve.png'))
    ax2.set_title('ROC Curve')
    ax2.axis('off')

plt.tight_layout()
plt.show()

## 7. Individual Image Prediction

In [ ]:
from predict import predict_single_image, format_prediction_output

test_df = pd.read_csv('outputs/splits/test.csv')
sample_row = test_df.iloc[0]
trained_model = tf.keras.models.load_model('checkpoints/best_model.h5', compile=False)

result = predict_single_image(
    image_path=sample_row['image_path'],
    model=trained_model,
    threshold=0.5,
    actual_label='Pneumonia' if sample_row['label'] == 1 else 'No Pneumonia'
)

print(format_prediction_output(result))

# Display the image alongside its prediction
plt.figure(figsize=(4, 4))
plt.imshow(Image.open(sample_row['image_path']), cmap='gray')
plt.title(f"Pred: {result['predicted_class']} ({result['predicted_probability']*100:.1f}%)\nActual: {result.get('actual_class', 'Unknown')}")
plt.axis('off')
plt.show()